# nano Beatrix v1: the lead body's plain screen run, resumed

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_lastfix_resume_pdc_bare.ipynb)

The first screen's notebook (`nano_v1_lastfix_screen.ipynb`) with one change: its arm list is the lead body's plain run alone (`pdc-bare`: the dual-hub Canon body as the v1 train ran it). That run stopped near step 4,000 of 4,960 when its Colab session ended; its saved state is in the repo, so C3 resumes it there and trains its last 960 steps (about 10 minutes on an RTX PRO 6000), then reads the last block's direction at 4,500 and 4,960, the drift memory on random bytes, and writes its record (`v1_lastfix/pdc-bare/screen_record.json`). The reads before step 4,000 are in its TensorBoard file in the repo.

It completes the comparison of the lead body's output layer on AdamW (`pdc-head-adamw0.1`, `pdc-head-adamw1.2`) with its plain run at the end of the screen.

Run C1 to C3 in order (C4 reads the curves and tables any time).


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.28"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import lastfix_screen as LS
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 28), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK ============
OUT = '/content/nano_v1_lastfix'                      # the session folder: the starts, the runs, the records
REPO = LS.REPO                                        # AbstractPhil/alephllm-nano-beatrix-training, under v1_lastfix/<arm>/
NAMES = ['pdc-bare']                                  # the lead body's plain run alone: it resumes from its saved state near step 4,000
MAX_HOURS = None                                      # this session's budget in hours (None: until every arm is done)
print('arms:', ', '.join(NAMES), '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: every arm's start (checked against a fresh build), the length, the push probe. Trains nothing.
import json, time
births = {n: LS.write_arm_birth(LS.BY_NAME[n], f'{OUT}/births') for n in NAMES}
print('starts:', ', '.join(f"{n} <- {LS.BY_NAME[n]['craft']} seed {LS.BY_NAME[n]['seed']} {LS.BY_NAME[n]['mode'] or 'bare'}" for n in NAMES))
step_s = {'nano-castor-v1': 0.299, 'nano-pollux-dual-canon-v1': 0.493}      # the v1 bench on an RTX PRO 6000 at 16 x 4 (2026-10-14)
mins = {n: (4960 * step_s[LS.BY_NAME[n]['craft']] + 12 * 25) / 60 for n in NAMES}
print(f'-> about {sum(mins.values()) / 60:.1f} h on an RTX PRO 6000 (training plus a full eval and the gauge every 500 steps)')
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'{LS.PREFIX}/preflight/push_probe.json'
    api.upload_file(path_or_fileobj=json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                                'card': torch.cuda.get_device_name(0), 'arms': NAMES}).encode(),
                    path_in_repo=path, repo_id=REPO, commit_message='v1 last-write screen: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: C3 runs locally and ships nothing')

In [ ]:
# C3 (T) — THE SCREEN: arm by arm; the trainer's own lines and the screen's gauge lines ([lastfix]) print as they come. Rerun the cell to
# continue (a new session too): each arm resumes from the repo, finished arms are read back. The stop button saves the running arm first.
RECS = LS.run(OUT, HF_TOKEN if PUSH_OK else None, NAMES, max_hours=MAX_HOURS)
from IPython.display import Markdown, display
display(Markdown('**At the end of the warmup (step 1,145)**\n\n' + LS.table(RECS, 1145)))
display(Markdown('**At the end (each arm\'s last read)**\n\n' + LS.table(RECS)))

In [ ]:
# C4 — THE CURVES AND THE TABLES from the repo (every session; tokenless: this session's files). Trains nothing.
import json
from IPython.display import Markdown, display
recs, logdir = {}, f'{OUT}/runs'
if HF_TOKEN:
    from huggingface_hub import snapshot_download
    hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                            allow_patterns=[f'{LS.PREFIX}/{n}/runs/*' for n in NAMES] + [f'{LS.PREFIX}/{n}/screen_record.json' for n in NAMES])
    logdir = f'{hub}/{LS.PREFIX}'
    for n in NAMES:
        p = f'{hub}/{LS.PREFIX}/{n}/screen_record.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
else:
    for n in NAMES:
        p = f'{OUT}/lastfix/{n}.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
display(Markdown('**At the end of the warmup (step 1,145)**\n\n' + LS.table(recs, 1145)))
display(Markdown('**Each arm\'s last read**\n\n' + LS.table(recs)))
%load_ext tensorboard
%tensorboard --logdir {logdir}

### Reading the result

| read | bare (the craft as the v1 train ran it) | a form that works |
|---|---|---|
| share on the last block's direction | climbs toward 0.9+ | stays well under bare at every read |
| the last bank's raw write (before the bound) | grows tens of times | may still grow: the bound acts after it, so the share above is the read that counts |
| the last block's rank | falls to ~13 of 256 | stays higher |
| bits a byte at 4,960 | the reference | within max(0.001, the bare seeds' spread) |
| the exit's H on random bytes | the reference | inside bare's spread (the drift kept) |

The form that passes goes into the v1 crafts (a config flag, one line in the v1 notebook), and the v1 train starts from scratch.

### Changing the screen
| to | the edit in C1 |
|---|---|
| only the Castor arms | `NAMES = [n for n in NAMES if n.startswith('c-')]` |
| a session budget | `MAX_HOURS = 2` |